# SerendibAI Gemma 4 E4B Q4 – Colab inference dashboard

This notebook runs the production `Q4_0` GGUF using CUDA llama.cpp and exposes a temporary public Gradio chat dashboard. It runs text inference only.

Before running, select **Runtime → Change runtime type → GPU**. A free Colab T4 (16 GB) is sufficient. You must first accept the Gemma terms on Hugging Face. Add a read token named `HF_TOKEN` in **Colab Secrets** and grant this notebook access to it, or enter the token at the hidden prompt. The token is kept only in the active Colab runtime.

In [ ]:
!nvidia-smi
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --no-cache-dir --force-reinstall --no-binary llama-cpp-python llama-cpp-python
!pip install -q --no-cache-dir gradio huggingface_hub

In [ ]:
import os
from getpass import getpass
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except ImportError:
    hf_token = None
except userdata.SecretNotFoundError:
    hf_token = None

if not hf_token:
    hf_token = getpass("Hugging Face token (input stays hidden): " )
if not hf_token:
    raise RuntimeError("An HF_TOKEN with Gemma access is required.")

os.environ["HF_TOKEN"] = hf_token
MODEL_REPO = "google/gemma-4-E4B-it-qat-q4_0-gguf"
MODEL_FILE = "gemma-4-E4B_q4_0-it.gguf"
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, token=hf_token)

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    n_batch=512,
    verbose=False,
)
print(f"Loaded {MODEL_REPO} with all layers on the Colab GPU.")

In [ ]:
import gradio as gr

DEFAULT_SYSTEM_PROMPT = "You are a helpful, concise assistant."

def respond(message, history, system_prompt, temperature, max_tokens):
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(
        {"role": item["role"], "content": item["content"]}
        for item in history
        if item["role"] in {"user", "assistant"}
    )
    messages.append({"role": "user", "content": message})

    answer = ""
    for chunk in llm.create_chat_completion(
        messages=messages,
        temperature=float(temperature),
        max_tokens=int(max_tokens),
        stream=True,
    ):
        answer += chunk["choices"][0]["delta"].get("content", "")
        yield answer

with gr.Blocks(title="SerendibAI Gemma 4 E4B Q4") as demo:
    gr.Markdown("# SerendibAI Gemma 4 E4B Q4\nCUDA llama.cpp on a Colab GPU. The shared link is temporary and public—do not enter secrets or customer data.")
    with gr.Accordion("Generation settings", open=False):
        system_prompt = gr.Textbox(value=DEFAULT_SYSTEM_PROMPT, label="System prompt", lines=3)
        with gr.Row():
            temperature = gr.Slider(0, 1.5, value=0.2, step=0.05, label="Temperature")
            max_tokens = gr.Slider(16, 512, value=160, step=16, label="Max new tokens")
    gr.ChatInterface(
        fn=respond,
        additional_inputs=[system_prompt, temperature, max_tokens],
        type="messages",
        examples=["Hello! Please introduce yourself in Sinhala.", "What can you help me with?"],
    )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)